# Class 비율

In [ ]:
import os
import matplotlib.pyplot as plt
data = []
path = './train'
for filename in sorted(os.listdir("./train")):
    data.append(len(os.listdir(os.path.join(path, filename))))
    if data[-1] < 70:
        print(filename, data[-1])

plt.plot(data)

# 이미지 해상도

In [ ]:
import cv2, glob, tqdm
import numpy as np
# datas = []
filenames = []
res = []
for filename in tqdm.tqdm(glob.glob("./train/*/*.jpg")):
    filename = filename.replace("\\", "/")
    image_array = np.fromfile(filename, np.uint8)
    image = cv2.imdecode(image_array, cv2.IMREAD_COLOR)
    # datas.append(image)
    res.append(image.shape)
    filenames.append(filename)
res = np.array(res, dtype=np.float32)
ratio = res[:, 0] / res[:, 1]

100%|██████████| 33137/33137 [00:47<00:00, 691.05it/s]


In [ ]:
res

array([[428., 822.,   3.],
       [433., 553.,   3.],
       [361., 784.,   3.],
       ...,
       [423., 769.,   3.],
       [434., 725.,   3.],
       [417., 516.,   3.]], dtype=float32)

In [ ]:
ratio

array([0.52068126, 0.7830018 , 0.46045917, ..., 0.55006504, 0.5986207 ,
       0.80813956], dtype=float32)

In [ ]:
import shutil, os
shutil.rmtree("./selected")
os.makedirs("./selected")

for i, r in enumerate(ratio):
    basename = filenames[i].split("/")[-1]
    if r <0.3: 
        print(filenames[i])

        shutil.copy(filenames[i], os.path.join("./selected", f"{r}_{basename}"))


./test/TEST_00425.jpg
./test/TEST_00530.jpg
./test/TEST_00751.jpg
./test/TEST_00780.jpg
./test/TEST_00992.jpg
./test/TEST_01063.jpg
./test/TEST_01140.jpg
./test/TEST_01255.jpg
./test/TEST_01505.jpg
./test/TEST_01782.jpg
./test/TEST_01817.jpg
./test/TEST_01942.jpg
./test/TEST_02003.jpg
./test/TEST_02049.jpg
./test/TEST_02578.jpg
./test/TEST_02653.jpg
./test/TEST_03003.jpg
./test/TEST_03104.jpg
./test/TEST_03453.jpg
./test/TEST_03960.jpg
./test/TEST_04005.jpg
./test/TEST_04340.jpg
./test/TEST_04419.jpg
./test/TEST_04467.jpg
./test/TEST_04718.jpg
./test/TEST_04785.jpg
./test/TEST_04819.jpg
./test/TEST_04827.jpg
./test/TEST_04910.jpg
./test/TEST_05738.jpg
./test/TEST_05754.jpg
./test/TEST_05945.jpg
./test/TEST_06117.jpg
./test/TEST_06157.jpg
./test/TEST_06186.jpg
./test/TEST_06229.jpg
./test/TEST_06240.jpg
./test/TEST_06317.jpg
./test/TEST_06487.jpg
./test/TEST_06512.jpg
./test/TEST_06544.jpg
./test/TEST_06921.jpg
./test/TEST_06996.jpg
./test/TEST_07125.jpg
./test/TEST_07254.jpg
./test/TES

In [ ]:
import cv2, glob, tqdm
import numpy as np
# datas = []
filenames = []
res = []
for filename in tqdm.tqdm(glob.glob("./test/*.jpg")):
    filename = filename.replace("\\", "/")
    image_array = np.fromfile(filename, np.uint8)
    image = cv2.imdecode(image_array, cv2.IMREAD_COLOR)
    # datas.append(image)
    res.append(image.shape)
    filenames.append(filename)
res = np.array(res, dtype=np.float32)
ratio = res[:, 0] / res[:, 1]

100%|██████████| 8258/8258 [00:52<00:00, 156.23it/s]


# K-Fold index 설정

In [ ]:
import os, tqdm
import numpy as np
root_dir='./dataset/data/train'

samples = []

classes = sorted(os.listdir(root_dir))
class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}

data = [[] for _ in range(len(classes))]

for cls_idx, cls_name in enumerate(tqdm.tqdm(classes, desc='Processing')):
    cls_folder = os.path.join(root_dir, cls_name)
    for fname in sorted(os.listdir(cls_folder)):
        if fname.lower().endswith(('.jpg')):
            image_path = os.path.join(cls_folder, fname)
            target = cls_idx
            data[cls_idx].append([image_path, target])

Processing: 100%|██████████| 396/396 [00:00<00:00, 1605.12it/s]


In [ ]:
k = 3
def split_list(lst, n):
    k, m = divmod(len(lst), n)
    return [lst[i*k + min(i, m):(i+1)*k + min(i+1, m)] for i in range(n)]

rng = np.random.RandomState(42)
folded = [split_list(rng.permutation(range(len(data[i]))).tolist(), n=k) for i in range(len(data))] # 각 파트는 Validation index들을 의미함
print(folded)

[[[75, 0, 70, 22, 12, 56, 10, 18, 4, 67, 61, 64, 53, 73, 62, 66, 33, 78, 54, 72, 11, 30, 40, 28, 9, 65, 5, 39, 31], [35, 45, 44, 16, 42, 34, 7, 49, 82, 19, 83, 25, 47, 13, 24, 3, 17, 38, 8, 68, 6, 55, 36, 77, 85, 43, 50, 46, 84], [15, 69, 27, 41, 58, 26, 48, 76, 57, 32, 81, 59, 63, 79, 37, 29, 1, 52, 21, 2, 23, 80, 74, 20, 60, 71, 14, 51]], [[80, 42, 61, 60, 49, 20, 41, 15, 17, 77, 5, 82, 78, 38, 31, 9, 1, 56, 24, 37, 71, 45, 58, 29, 55, 65, 3, 19], [16, 63, 30, 18, 12, 50, 35, 79, 72, 48, 51, 25, 66, 46, 2, 13, 53, 57, 74, 52, 26, 21, 39, 43, 68, 36, 23, 22], [54, 47, 67, 33, 11, 76, 75, 6, 27, 81, 4, 32, 69, 34, 73, 10, 62, 7, 0, 8, 70, 64, 44, 14, 28, 40, 59]], [[66, 45, 9, 73, 47, 14, 20, 17, 13, 12, 29, 33, 7, 68, 46, 41, 6, 30, 35, 55, 24, 28, 72, 42, 39, 15, 34], [21, 37, 40, 19, 4, 67, 59, 10, 3, 60, 5, 44, 25, 75, 18, 0, 32, 65, 53, 22, 27, 49, 56, 52, 16, 54], [70, 62, 77, 1, 38, 11, 64, 57, 63, 74, 48, 51, 31, 58, 23, 43, 50, 36, 78, 8, 26, 71, 69, 2, 76, 61]], [[74, 26, 25,

# GradCAM 분석

In [ ]:
import torch
import cv2
import numpy as np
from utils import load_model
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image, preprocess_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
device = 'cuda:0'


In [ ]:
import torch.nn as nn
class Wraper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model=model

    def forward(self, x):
        return self.model(x)['output']


## 단일, 다중 모델 GradCAM 분석

In [ ]:
from utils import load_model, imread_unicode
import glob, tqdm
import numpy as np
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import os
model_lst = [
    './experiments/45_-1/best.pt',
    # './experiments/12/best.pt',
]

models = [load_model(path, device=device)[0].eval() for path in model_lst]
wrapers = [Wraper(model) for model in models]
target_layer = [wraper.model.backbone.layer4[-1] for wraper in wrapers]
save_dir = './grad_cam'

In [ ]:
from cfg import ignore_datas

In [ ]:
root_dir='./dataset/data/train'
classes = sorted(os.listdir(root_dir))
class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
os.makedirs(save_dir, exist_ok=True)
for filename in tqdm.tqdm(glob.glob("./dataset/data/train/*/*.jpg")):
    classname = filename.split("/")[-2]
    classidx = class_to_idx[classname]
    basename = filename.split("/")[-1]
    if basename not in ignore_datas:
        image = imread_unicode(filename)
        image = cv2.resize(image, wrapers[0].model.wh)
        image_ori = image.copy()
        image = (image- wrapers[0].model.mean )/ wrapers[0].model.std
        image = np.transpose(image, (2, 0, 1))
        image = image[None]
        image = torch.from_numpy(image).float().to(device)
        outputs = np.zeros((image_ori.shape[0], image_ori.shape[1]*len(wrapers), 3), dtype=np.uint8)
        with torch.no_grad():
            pred = np.argmax(models[0](image)['output'].detach().cpu().numpy(), axis=1)[0]
        if pred!= classidx:
            for i in range(len(wrapers)):
                cam = GradCAM(model=wrapers[i], target_layers=[target_layer[i]])


                # CAM 계산
                grayscale_cam = cam(input_tensor=image, targets=[ClassifierOutputTarget(pred)])[0]  # 첫 번째 이미지
                visualization = show_cam_on_image(image_ori.astype(np.float32)/255.0, grayscale_cam)
                visualization = cv2.cvtColor(visualization, cv2.COLOR_RGB2BGR)
                outputs[:, i*image_ori.shape[1]:(i+1)*image_ori.shape[1]] = visualization
            

            # 저장 또는 출력
            cv2.imwrite(os.path.join(save_dir, f'{basename}'), outputs)


In [ ]:
coarse_dropout(image, num_holes_range=(1, 2), hole_height_range=(0.1, 0.3), hole_width_range=(0.1, 0.3), value=(255, 255, 255)) # 0.5

            
color_jitter(image, brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1) # 0.5

image[..., np.random.choice([0, 1, 2], size=3, replace=False) ] # rgb shift 0.5
            
rotate(image, np.random.uniform(-1, 1)*180) # 0.1
randomcrop(image, min_ratio=.5) # 0.25

# mixprob 0.25
cutmix # 0.5 alpha 1.0
mixup # 0.5 alpha 0.8

## Kfold를 통한 전체 데이터 분석

In [ ]:
import glob, tqdm
from cfg import folded_5, mean, std, ignore_datas
import os, tqdm
import numpy as np
root_dir='./dataset/data/train'

samples = []

classes = sorted(os.listdir(root_dir))
class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}

data = [[] for _ in range(len(classes))]

for cls_idx, cls_name in enumerate(tqdm.tqdm(classes, desc='Processing')):
    cls_folder = os.path.join(root_dir, cls_name)
    for fname in sorted(os.listdir(cls_folder)):
        if fname.lower().endswith(('.jpg')):
            image_path = os.path.join(cls_folder, fname)
            target = cls_idx
            data[cls_idx].append([image_path, target])

Processing:  57%|█████▋    | 225/396 [00:00<00:00, 2242.86it/s]

Processing: 100%|██████████| 396/396 [00:00<00:00, 2424.44it/s]


In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import os
save_dir = './gradcam'
os.makedirs(os.path.join(save_dir, 'total'), exist_ok=True)
for fold_num in range(5):
    model = load_model(f'./experiments/22_{fold_num}/best.pt', device=device, ema=False)[0].eval()
    wraper = Wraper(model)
    target_layer = wraper.model.backbone.layer4[-1]
    for cls_idx in tqdm.tqdm(range(len(folded_5))):
        for idx in sorted(folded_5[cls_idx][fold_num]):
            basename = data[cls_idx][idx][0].split("/")[-1].split(".")[0]
            if basename+".jpg" not in ignore_datas:
                image = cv2.imread(data[cls_idx][idx][0])
                image = cv2.resize(image, wraper.model.wh)
                image_ori = image.copy()
                image = (image-mean )/std
                image = np.transpose(image, (2, 0, 1))
                image = image[None]
                image = torch.from_numpy(image).float().to(device)
                with torch.no_grad():
                    pred = np.argmax(model(image)['output'].detach().cpu().numpy(), axis=1)[0]
                if pred!=cls_idx:

                    os.makedirs(os.path.join(save_dir, classes[cls_idx]), exist_ok=True)
                    cam = GradCAM(model=wraper, target_layers=[target_layer])


                    # CAM 계산
                    grayscale_cam = cam(input_tensor=image, targets=[ClassifierOutputTarget(pred)])[0]  # 첫 번째 이미지
                    visualization = show_cam_on_image(image_ori.astype(np.float32)/255.0, grayscale_cam)

                    # 저장 또는 출력
                    cv2.imwrite(os.path.join(save_dir, classes[cls_idx], f'{basename}_{fold_num}_{classes[pred]}.jpg'), cv2.cvtColor(visualization, cv2.COLOR_RGB2BGR))
                    cv2.imwrite(os.path.join(save_dir, 'total', f'{basename}_{fold_num}_{classes[pred]}.jpg'), cv2.cvtColor(visualization, cv2.COLOR_RGB2BGR))

# Detection model 분석

In [ ]:
# ddq 파라미터
import numpy as np
class_names = ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic_light', 'fire_hydrant', 'stop_sign', 'parking_meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports_ball', 'kite', 'baseball_bat', 'baseball_glove', 'skateboard', 'surfboard', 'tennis_racket', 'bottle', 'wine_glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot_dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted_plant', 'bed', 'dining_table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell_phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy_bear', 'hair_drier', 'toothbrush']

car_names = ['bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat']

car_idxs = np.array([i for i, c in enumerate(class_names) if c in car_names])

In [ ]:


def processing_ddq_result(image, scores, bboxes, labels, t_score=0.25, t_area=0.3):
    h, w, c = image.shape
    over = scores>=t_score
    num_bbox = over.sum()
    ret = {
        'vehicle': None,
        'else': {
            'erase': [],
            'remain': []
        }
        }
    filter_type = None
    if num_bbox == 0:
        ret['vehicle'] = [0, 0, w, h]
        filter_type = 0
    else :
        scores = scores[over]
        bboxes = bboxes[over]
        labels = labels[over]
        box_w = bboxes[:, 2] -bboxes[:, 0]
        box_h = bboxes[:, 3] -bboxes[:, 1]
        bboxes = bboxes.astype(np.int64)
        box_area = box_h*box_w
        image_area = h*w
        area_ratio = box_area/image_area

        
        is_valid_area = area_ratio >= t_area
        num_valid_area = is_valid_area.sum()
        num_not_valid_area = (is_valid_area==False).sum()

        is_vehicles = np.array([True if c in car_idxs else False for c in labels ])
        unique_vehicles = np.unique(labels[is_vehicles])
        
        if num_valid_area == 0 :
            if num_not_valid_area == 0: # 진짜 아무것도 없는 케이스, 이 케이스는 num_bbox 조건문 부분에서 잡힘
                ret['vehicle'] = [0, 0, w, h]
                filter_type = 1
            else: # 객체가 있긴 한데 작은 케이스
                if (~is_valid_area & is_vehicles).sum() > 0: # 차량이 있는경우 제일 큰 BBOX 설정
                    max_area_idx = np.argmax(area_ratio[~is_valid_area & is_vehicles])
                    ret['vehicle'] = bboxes[~is_valid_area & is_vehicles][max_area_idx].tolist()
                    for i in range(len(area_ratio[~is_valid_area & is_vehicles])):
                        if i != max_area_idx:
                            ret['else']['erase'].append(bboxes[~is_valid_area & is_vehicles][i].tolist())
                    filter_type = 2
                else:
                    ret['vehicle'] = [0, 0, w, h]
                    filter_type = 3

                for i in range(len(~is_valid_area & ~is_vehicles)): # 차량이 아닌경우
                    ret['else']['remain'].append(bboxes[i].tolist())
        else :
            if num_not_valid_area > 0:
                for i in range(len(bboxes)):
                    if is_valid_area[i]==False:
                        if is_vehicles[i] :
                            ret['else']['erase'].append(bboxes[i].tolist())
                        else:
                            ret['else']['remain'].append(bboxes[i].tolist())
                
            if num_valid_area == 1:
                ret['vehicle'] = bboxes[is_valid_area].tolist()[0]
                filter_type = 4
            else:
                max_area_idx = np.argmax(area_ratio[is_valid_area])
                ret['vehicle'] = bboxes[is_valid_area][max_area_idx].tolist()
                filter_type = 5

    ret['type'] = filter_type

    return ret

## Train set 처리

In [ ]:
import pickle
with open("./dataset/detection/inference_result/inference_results_train_ddq.pkl", mode='rb') as f:
    datas = pickle.load(f)

In [ ]:
datas.keys()

dict_keys(['1시리즈_F20_2013_2015', '1시리즈_F20_2016_2019', '1시리즈_F40_2020_2024', '2008_2015_2017', '2시리즈_그란쿠페_F44_2020_2024', '2시리즈_액티브_투어러_F45_2019_2021', '2시리즈_액티브_투어러_U06_2022_2024', '3008_2세대_2018_2023', '3시리즈_E90_2005_2012', '3시리즈_F30_2013_2018', '3시리즈_G20_2019_2022', '3시리즈_G20_2023_2025', '3시리즈_GT_F34_2014_2021', '4시리즈_F32_2014_2020', '4시리즈_G22_2021_2023', '4시리즈_G22_2024_2025', '5008_2세대_2018_2019', '5008_2세대_2021_2024', '5시리즈_F10_2010_2016', '5시리즈_G30_2017_2023', '5시리즈_G60_2024_2025', '5시리즈_GT_F07_2010_2017', '6시리즈_F12_2011_2018', '6시리즈_GT_G32_2018_2020', '6시리즈_GT_G32_2021_2024', '718_박스터_2017_2024', '718_카이맨_2017_2024', '7시리즈_F01_2009_2015', '7시리즈_G11_2016_2018', '7시리즈_G11_2019_2022', '7시리즈_G70_2023_2025', '8시리즈_G15_2020_2024', '911_2003_2019', '911_992_2020_2024', 'A4_B9_2016_2019', 'A4_B9_2020_2024', 'A5_F5_2019_2024', 'A6_C8_2019_2025', 'A7_2012_2016', 'A7_4K_2020_2024', 'A8_D5_2018_2023', 'AMG_GT_2016_2024', 'A_클래스_W176_2015_2018', 'A_클래스_W177_2020_2025', 'All_New_XJ_2016_2019'

In [ ]:
datas['1시리즈_F20_2013_2015'].keys()

dict_keys(['1시리즈_F20_2013_2015_0000.jpg', '1시리즈_F20_2013_2015_0001.jpg', '1시리즈_F20_2013_2015_0002.jpg', '1시리즈_F20_2013_2015_0003.jpg', '1시리즈_F20_2013_2015_0004.jpg', '1시리즈_F20_2013_2015_0005.jpg', '1시리즈_F20_2013_2015_0006.jpg', '1시리즈_F20_2013_2015_0007.jpg', '1시리즈_F20_2013_2015_0008.jpg', '1시리즈_F20_2013_2015_0009.jpg', '1시리즈_F20_2013_2015_0010.jpg', '1시리즈_F20_2013_2015_0011.jpg', '1시리즈_F20_2013_2015_0012.jpg', '1시리즈_F20_2013_2015_0013.jpg', '1시리즈_F20_2013_2015_0014.jpg', '1시리즈_F20_2013_2015_0015.jpg', '1시리즈_F20_2013_2015_0016.jpg', '1시리즈_F20_2013_2015_0017.jpg', '1시리즈_F20_2013_2015_0018.jpg', '1시리즈_F20_2013_2015_0019.jpg', '1시리즈_F20_2013_2015_0020.jpg', '1시리즈_F20_2013_2015_0021.jpg', '1시리즈_F20_2013_2015_0022.jpg', '1시리즈_F20_2013_2015_0023.jpg', '1시리즈_F20_2013_2015_0024.jpg', '1시리즈_F20_2013_2015_0025.jpg', '1시리즈_F20_2013_2015_0026.jpg', '1시리즈_F20_2013_2015_0027.jpg', '1시리즈_F20_2013_2015_0028.jpg', '1시리즈_F20_2013_2015_0029.jpg', '1시리즈_F20_2013_2015_0030.jpg', '1시리즈_F20_2013_2015_0031.jpg

In [ ]:
datas['1시리즈_F20_2013_2015']['1시리즈_F20_2013_2015_0000.jpg'].keys()

dict_keys(['bboxes', 'scores', 'labels'])

In [ ]:
import numpy as np
class_names = ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic_light', 'fire_hydrant', 'stop_sign', 'parking_meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports_ball', 'kite', 'baseball_bat', 'baseball_glove', 'skateboard', 'surfboard', 'tennis_racket', 'bottle', 'wine_glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot_dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted_plant', 'bed', 'dining_table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell_phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy_bear', 'hair_drier', 'toothbrush']

car_names = ['bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat']

car_idxs = np.array([i for i, c in enumerate(class_names) if c in car_names])

In [ ]:
import cv2, tqdm

result = {}
for cls_name, data in tqdm.tqdm(datas.items()):
    for filename, d in data.items():
        image_path = f'./dataset/data/train/{cls_name}/{filename}'
        image = imread_unicode(image_path)
        h, w, c = image.shape
        scores = d['scores'].copy()
        bboxes = d['bboxes'].copy()
        labels = d['labels'].copy()
        
        result[filename] = processing_ddq_result(image, scores, bboxes, labels, t_score=0.25, t_area=0.3)


100%|██████████| 396/396 [00:41<00:00,  9.53it/s]


In [ ]:
import json
with open("./dataset/detection/processed/train_det.json", mode='w') as f:
    json.dump(result, f)

## Trainset crop 위치 시각화

In [ ]:
import os, tqdm
import imageio

def toint(data):
    return [int(d) for d in data]

for k, v in tqdm.tqdm(result.items()):
    cls_name = k.split(".")[0][:-5]
    filter_type = v['type']
    
    image_path = f'./dataset/data/train/{cls_name}/{k}'
    image = imread_unicode(image_path)
    # vehicle bbox (초록색)
    vehicle_bbox = v['vehicle']
    
    x1, y1, x2, y2 = toint(vehicle_bbox)
    cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)  # 초록색

    # else > erase bbox (빨간색)
    for bbox in v.get('else', {}).get('erase', []):
        x1, y1, x2, y2 = toint(bbox)
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 2)  # 빨간색

    # 저장
    image = image[..., ::-1]

    os.makedirs(f'./dataset/data/vis_det_train/{filter_type}', exist_ok=True)
    save_path = f'./dataset/data/vis_det_train/{filter_type}/{k}'
    imageio.imwrite(save_path, image)

100%|██████████| 33137/33137 [06:12<00:00, 89.06it/s] 


## Testset 처리

In [ ]:
import pickle
with open("./dataset/detection/inference_result/inference_results_test_ddq.pkl", mode='rb') as f:
    datas = pickle.load(f)

In [ ]:
datas.keys()

dict_keys(['TEST_00000.jpg', 'TEST_00001.jpg', 'TEST_00002.jpg', 'TEST_00003.jpg', 'TEST_00004.jpg', 'TEST_00005.jpg', 'TEST_00006.jpg', 'TEST_00007.jpg', 'TEST_00008.jpg', 'TEST_00009.jpg', 'TEST_00010.jpg', 'TEST_00011.jpg', 'TEST_00012.jpg', 'TEST_00013.jpg', 'TEST_00014.jpg', 'TEST_00015.jpg', 'TEST_00016.jpg', 'TEST_00017.jpg', 'TEST_00018.jpg', 'TEST_00019.jpg', 'TEST_00020.jpg', 'TEST_00021.jpg', 'TEST_00022.jpg', 'TEST_00023.jpg', 'TEST_00024.jpg', 'TEST_00025.jpg', 'TEST_00026.jpg', 'TEST_00027.jpg', 'TEST_00028.jpg', 'TEST_00029.jpg', 'TEST_00030.jpg', 'TEST_00031.jpg', 'TEST_00032.jpg', 'TEST_00033.jpg', 'TEST_00034.jpg', 'TEST_00035.jpg', 'TEST_00036.jpg', 'TEST_00037.jpg', 'TEST_00038.jpg', 'TEST_00039.jpg', 'TEST_00040.jpg', 'TEST_00041.jpg', 'TEST_00042.jpg', 'TEST_00043.jpg', 'TEST_00044.jpg', 'TEST_00045.jpg', 'TEST_00046.jpg', 'TEST_00047.jpg', 'TEST_00048.jpg', 'TEST_00049.jpg', 'TEST_00050.jpg', 'TEST_00051.jpg', 'TEST_00052.jpg', 'TEST_00053.jpg', 'TEST_00054.jpg',

In [ ]:
datas['TEST_00000.jpg'].keys()

dict_keys(['bboxes', 'scores', 'labels'])

In [ ]:
import cv2, tqdm
t_score = 0.25
t_area = 0.3

result = {}
for filename, data in tqdm.tqdm(datas.items()):
    image_path = f'./dataset/data/test/{filename}'
    image = cv2.imread(image_path)
    scores = data['scores'].copy()
    bboxes = data['bboxes'].copy()
    labels = data['labels'].copy()
    result[filename] = processing_ddq_result(image, scores, bboxes, labels, t_score=0.25, t_area=0.3)



100%|██████████| 8258/8258 [00:16<00:00, 511.40it/s]


In [ ]:
import json
with open("./dataset/detection/processed/test_det.json", mode='w') as f:
    json.dump(result, f)

# 번호판 필터링

In [ ]:
import glob
import tqdm
import cv2
import numpy as np
import os
import imageio
def imread_unicode(path):
    with open(path, 'rb') as f:
        img_array = np.asarray(bytearray(f.read()), dtype=np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
        return img
    
for filename in tqdm.tqdm(glob.glob(f'./dataset/data/train/*/*.jpg')):
    filename = filename.replace('\\', '/')
    basename = filename.split("/")[-1]
    image = imread_unicode(filename)
    white_mask = np.all(image == [255, 255, 255], axis=-1)
    mask = (image/2).astype(np.uint8)
    mask[white_mask] = image[white_mask]
    os.makedirs(f'./masking/', exist_ok=True)
    save_path = f'./masking/{basename}'
    imageio.imwrite(save_path, mask)


# 데이터셋 테스트

In [ ]:
from dataset.dataset import HAI

ds = HAI(det_crop=True, erase_other_car=True, mode=0, aug=True)

In [ ]:
import matplotlib.pyplot as plt
data = ds.__getitem__(200, True)
plt.subplot(1, 2, 1)
plt.imshow(data['image'])
plt.subplot(1, 2, 2)
plt.imshow(data['mask'])
